# Actualización controlada de `Resultados.estado_verificacion_oc` — Agosto 2026

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Actualizar_Estado_Resultados_Agosto_Colab.ipynb)

**Destino único:** clon Google Sheets. **Única columna editable:** `J` (`estado_verificacion_oc`). No altera `validado`, proveedor, montos ni fecha. No toca producción, front ni ETL.

Requiere subir `simulacion_actualizacion_resultados_agosto_2026.csv` (65.945 filas), generado por la auditoría. **Ejecuta primero en simulación**; revisa los conteos. Solo después escribe la autorización exacta para efectuar la actualización. El notebook crea respaldo del clon antes de la primera escritura y verifica cada bloque.

**Estados son evidencia operativa provisional, no adjudicación jurídica verificada.** Ejecución debe ser exclusiva, sin otros escritores de la hoja.

In [ ]:
!pip -q install gspread google-api-python-client google-auth
from google.colab import auth, files
from google.auth import default
from googleapiclient.discovery import build
import gspread,csv,collections,datetime,time,hashlib,json
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds);drive=build('drive','v3',credentials=creds)
CLON='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
PRODUCCION='1y81QaPy37APkBwxlYrKyE66_1ECdhL87GyWNitTfHr4'
assert CLON!=PRODUCCION
ws=gc.open_by_key(CLON).worksheet('Resultados')
HEAD=['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc']
assert ws.row_values(1)==HEAD
print('Destino CLON validado, columna J exclusivamente')

In [ ]:
subidos=files.upload()
assert len(subidos)==1 and next(iter(subidos)).endswith('.csv'),'Sube únicamente el CSV de simulación'
archivo=next(iter(subidos))
with open(archivo,encoding='utf-8-sig',newline='') as f:
 lector=csv.DictReader(f)
 necesarios={'codigo','estado_verificacion_oc_propuesto','validado_propuesto','accion_sobre_montos'}
 assert necesarios.issubset(lector.fieldnames)
 propuestas=list(lector)
assert len(propuestas)==65945, 'El volumen no coincide con el corte aprobado'
validos={'OC_ACEPTADA_MONTO_COINCIDE','OC_ACEPTADA_RUT_COINCIDE_NOMBRE_DIFIERE','OC_ACEPTADA_MONTO_DIFIERE','OC_ACEPTADA_FUERA_CORTE','SIN_OC_ACEPTADA'}
assert all(r['estado_verificacion_oc_propuesto'] in validos and r['validado_propuesto']=='SIN_CAMBIOS' and r['accion_sobre_montos']=='NINGUNA' for r in propuestas)
codigos=[r['codigo'].strip().upper() for r in propuestas]
assert len(codigos)==len(set(codigos)),'CSV con códigos duplicados'
propuesta={r['codigo'].strip().upper():r['estado_verificacion_oc_propuesto'] for r in propuestas}
print('CSV validado:',len(propuesta),'estados:',dict(collections.Counter(propuesta.values())))

## Comprobación antes de escribir

Se leen solamente los códigos de columna A y los estados actuales de columna J. Todos los códigos deben coincidir con el CSV. El único valor actual permitido, además del estado propuesto en una posible reanudación, es `OC no verificada`. Un estado inesperado bloquea la ejecución.

In [ ]:
claves=ws.col_values(1)
estados=ws.col_values(10)
assert claves[0]=='codigo' and estados[0]=='estado_verificacion_oc'
assert len(claves)==65946, f'Cantidad inesperada en clon: {len(claves)-1}'
claves=[str(x).strip().upper() for x in claves[1:]]
estados=[str(estados[i]).strip() if i<len(estados) else '' for i in range(1,len(claves)+1)]
assert len(set(claves))==len(claves) and set(claves)==set(propuesta),'Diferencia entre códigos de clon y simulación'
incompatibles=[(i+2,k,v) for i,(k,v) in enumerate(zip(claves,estados)) if v not in ('OC no verificada',propuesta[k])]
assert not incompatibles, f'Se encontraron estados no previstos: {incompatibles[:5]}'
pendientes=sum(v!=propuesta[k] for k,v in zip(claves,estados))
print('Total:',len(claves),'Pendientes:',pendientes,'Ya actualizados:',len(claves)-pendientes)
print('Clasificación objetivo:',json.dumps(dict(collections.Counter(propuesta.values())),ensure_ascii=False))
print('Hasta aquí: SIN ESCRITURAS')

## Confirmación y actualización de la columna J

La autorización es independiente de la preparación. Tras activarla se vuelve a leer la hoja, se respalda el clon y luego se escriben solo segmentos de la columna J. Cada segmento se verifica mediante lectura posterior y se puede reanudar sin duplicar registros.

In [ ]:
AUTORIZACION='' # Escribir exactamente: AUTORIZO ACTUALIZAR SOLO ESTADO RESULTADOS EN CLON
if AUTORIZACION!='AUTORIZO ACTUALIZAR SOLO ESTADO RESULTADOS EN CLON':
 print('Simulación únicamente: no se ha escrito en el clon.')
else:
 assert ws.row_values(1)==HEAD
 actual_a=ws.col_values(1)
 actual_j=ws.col_values(10)
 assert [str(t).strip().upper() for t in actual_a[1:]]==claves,'Cambió el orden o código de la hoja: detener'
 nuevos_j=[str(actual_j[i]).strip() if i<len(actual_j) else '' for i in range(1,len(claves)+1)]
 assert all(v in ('OC no verificada',propuesta[k]) for k,v in zip(claves,nuevos_j)), 'Se detectó un estado conflictivo'
 respaldo=drive.files().copy(fileId=CLON,body={'name':'Respaldo ANTES estados Resultados '+datetime.datetime.now().isoformat()},fields='id,name,webViewLink').execute()
 print('RESPALDO creado:',respaldo.get('webViewLink',respaldo['id']))
 escritos=0
 for desde in range(0,len(claves),250):
  parte=claves[desde:desde+250]
  actuales=nuevos_j[desde:desde+len(parte)]
  deseados=[propuesta[k] for k in parte]
  if actuales==deseados:continue
  comienzo=desde+2;final=comienzo+len(parte)-1
  rango=f'J{comienzo}:J{final}'
  # Escribir UNICAMENTE J, nunca columnas A:I.
  ws.update(range_name=rango,values=[[x] for x in deseados],value_input_option='RAW')
  confirmacion=ws.get(rango,value_render_option='UNFORMATTED_VALUE')
  confirmacion=[str(r[0]) if r else '' for r in confirmacion]
  assert confirmacion==deseados, f'ERROR DE VERIFICACIÓN EN {rango}; DETENER'
  escritos+=sum(a!=b for a,b in zip(actuales,deseados))
  if escritos%5000<250 or final==len(claves)+1:print('Estados nuevos verificados:',escritos)
  time.sleep(0.25)
 revisado=ws.col_values(10)[1:]
 assert revisado==[propuesta[k] for k in claves],'La validación final no coincide'
 print('ACTUALIZACION FINALIZADA:',escritos,'estados modificados; total comprobado:',len(claves))
 print('Respaldo ID:',respaldo['id'])